In [1]:
import tidy3d as td
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib widget
from matplotlib.colors import LogNorm

# import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import gc
import os
api_key = os.environ.get("TIDY3D_API_KEY")
import tidy3d.web as web
# web.configure("_blank_")
web.configure(api_key)

Configured successfully.


In [2]:
web.test()

12:01:21 UTC Authentication configured successfully!

In [3]:
h_planck = 6.62607015e-34
e_charge = 1.602176634e-19
def hz_to_ev(f): return (h_planck * f) / e_charge
def ev_to_hz(E): return (E * e_charge) / h_planck

In [4]:
save_dir = '/app/local_project/20Jan dimer holes'
os.makedirs(save_dir, exist_ok=True)

name = 'dipolEz_D200L100' + '_dimer_OUT'
os.makedirs('field_plots_'+ name, exist_ok=True)

In [5]:
def extract_monitor_data(filepath, monitor_name):
    
    # Load full simulation (unavoidable)
    sim_data = td.SimulationData.from_file(filepath)
    
    # Extract ONLY the monitor we need
    monitor_data = sim_data[monitor_name]
    
    # Delete the full simulation data immediately
    del sim_data
    gc.collect()
    
    return monitor_data

In [6]:
def get_plane_config(plane):
    """Return slicing info, labels, and field pairs for a given plane."""
    configs = {
        # ============ CHANGED: H-field plane configurations ============
        'Hxy': {'slicer': {'z': 0},
            'U_field': 'Hx', 'V_field': 'Hy',
            'xlabel': 'x (nm)', 'ylabel': 'y (nm)',
            'coord_keys': ('x', 'y')},
        
        'Hxz': {'slicer': {'y': 0},
            'U_field': 'Hx', 'V_field': 'Hz',
            'xlabel': 'x (nm)', 'ylabel': 'z (nm)',
            'coord_keys': ('x', 'z')},
        
        'Hyz': {'slicer': {'x': 0},
            'U_field': 'Hy', 'V_field': 'Hz',
            'xlabel': 'y (nm)', 'ylabel': 'z (nm)',
            'coord_keys': ('y', 'z')},
        # ============ END CHANGED ============
    }
    if plane not in configs:
        raise ValueError(f"plane must be one of {list(configs.keys())}")
    
    return configs[plane]


# ========================================


In [7]:
def get_field_component(monitor_data, monitor_data0, comp, slicer, peak, normalize):
    """Extract field component from monitor data"""
    data = getattr(monitor_data, comp).isel(**slicer).interp(f=peak)
    data0 = getattr(monitor_data0, comp).isel(**slicer).interp(f=peak)
    
    if normalize:
        return (data - data0) / data0
    else:
        return data - data0


def get_coords(monitor_data, plane):
    """Extract and format coordinates for plotting"""
    coord_key1, coord_key2 = get_plane_config(plane)['coord_keys']
    
    # Extract coordinates and convert to nm
    coord1 = monitor_data.Hx.coords[coord_key1].values * 1e3
    coord2 = monitor_data.Hx.coords[coord_key2].values * 1e3
    
    # Ensure increasing order
    if coord1[0] > coord1[-1]:
        coord1 = coord1[::-1]
    if coord2[0] > coord2[-1]:
        coord2 = coord2[::-1]
    
    return coord1, coord2


# ========================================


In [8]:
def prepare_Hfield_data(Hx, Hy, Hz, coord1, coord2, plane):
    """Compute |H| and meshgrid from field components"""
    # Convert to real NumPy arrays
    Hx = np.real(np.array(Hx))
    Hy = np.real(np.array(Hy))
    Hz = np.real(np.array(Hz))
    
    # Expected shape
    expected_shape = (len(coord2), len(coord1))
    
    # Transpose if needed
    if Hx.shape != expected_shape:
        print(f"(Hx.shape={Hx.shape}, expected={expected_shape})")
        Hx, Hy, Hz = Hx.T, Hy.T, Hz.T
    
    # For xy plane, apply additional transpose
    if plane == 'Hxy':
        Hx, Hy, Hz = Hx.T, Hy.T, Hz.T
        print("[INFO] Applied xy-plane transpose for correct orientation")
    
    # Compute magnitude
    H = np.sqrt(np.abs(Hx)**2 + np.abs(Hy)**2 + np.abs(Hz)**2)
    
    # Build coordinate mesh
    horizontal_axis, vertical_axis = np.meshgrid(coord1, coord2)
    
    # Orientation diagnostics
    print(f"[DEBUG] {plane}-plane orientation check:")
    print(f"  horizontal_axis shape={horizontal_axis.shape}, "
          f"vertical_axis shape={vertical_axis.shape}")
    print(f"  H-field array shape={H.shape}")
    
    return H, horizontal_axis, vertical_axis, Hx, Hy, Hz

In [9]:
def plot_Hfield(H, horizontal_axis, vertical_axis, U, V, 
               xlabel, ylabel, plane, peak, name, monitor,
               fixed_scale, density, arrow, save_path=None):
    """Render H-field magnitude and streamlines"""
    plt.figure(figsize=(8, 6))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"{name} | {plane}-{monitor} | {hz_to_ev(peak):.3f} eV")
    
    # Color scale
    if fixed_scale:
        cmap_data = H
        cbar_label = '|H| induced'  # CHANGED: Label for H-field
        vmin, vmax = np.percentile(H, [1, 99])
    else:
        cmap_data = np.abs(H) * 100
        cbar_label = '|H| (%)'
        vmin, vmax = 0, 1000
    
    # Draw color map
    plt.pcolor(horizontal_axis, vertical_axis, cmap_data, 
               cmap='viridis',  # CHANGED: Different colormap for H-field
               shading='auto',
               norm=LogNorm(vmin=vmin, vmax=100*vmax))  
    plt.colorbar(label=cbar_label)
    
    # Streamlines
    plt.streamplot(horizontal_axis, vertical_axis, U, V,
                   density=density,
                   linewidth=(H - H.min()) / (H.max() - H.min()) + 0.05,
                   color='white',
                   arrowstyle=arrow)
    
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Saved: {os.path.basename(save_path)}")
    
    plt.close()

In [10]:
def plot_Hfield_stream(monitor_data, monitor_data0, monitor, peak, plane='Hxz', 
                      normalize=False, fixed_scale=True,
                      density=2.5, arrow='fancy, head_length=0.7',
                      save_path=None):
    """Orchestrate H-field extraction and plotting for a 2D plane"""
    cfg = get_plane_config(plane)
    
    # Extract H-field components (CHANGED: Hx, Hy, Hz instead of Ex, Ey, Ez)
    Hx = get_field_component(monitor_data, monitor_data0, 'Hx', cfg['slicer'], peak, normalize)
    Hy = get_field_component(monitor_data, monitor_data0, 'Hy', cfg['slicer'], peak, normalize)
    Hz = get_field_component(monitor_data, monitor_data0, 'Hz', cfg['slicer'], peak, normalize)
    
    # Get coordinates and prepare data
    coord1, coord2 = get_coords(monitor_data, plane)
    H, X, Y, Hx, Hy, Hz = prepare_Hfield_data(Hx, Hy, Hz, coord1, coord2, plane)
    
    # Select vector components for streamlines (CHANGED: Hx, Hy, Hz)
    field_map = {'Hx': Hx, 'Hy': Hy, 'Hz': Hz}
    U = field_map[cfg['U_field']]
    V = field_map[cfg['V_field']]
    
    # Plot
    plot_Hfield(H, X, Y, U, V, cfg['xlabel'], cfg['ylabel'], plane, peak, 
               name, monitor, fixed_scale, density, arrow, save_path=save_path)

    

In [11]:
def process_single_monitor(monitor_name, peak, plane, save_path):

    try:
        # Extract only this monitor from both files
        monitor_data = extract_monitor_data(f'{save_dir}/{name}.hdf5', monitor_name)
        monitor_data0 = extract_monitor_data(f'{save_dir}/{name}_empty.hdf5', monitor_name)
        
        # ============ CHANGED: Call plot_Hfield_stream instead of plot_Efield_stream ============
        # Process and plot H-field
        plot_Hfield_stream(
            monitor_data, monitor_data0,
            monitor=monitor_name,
            peak=peak,
            plane=plane,
            normalize=False,
            save_path=save_path
        )
        # ============ END CHANGED ============
        
        # Free memory
        del monitor_data, monitor_data0
        gc.collect()
        
        print(f"✓ Completed {monitor_name}\n")
        return True

    except KeyError:
        print(f"⚠ Monitor '{monitor_name}' not found in simulation data - SKIPPING")
        print(f"{'='*60}\n")
        return False
    except Exception as e:
        print(f"❌ Error processing {monitor_name}: {e}")
        print(f"{'='*60}\n")
        return False
    

In [ ]:
# ========= # Visualisation # =============
# sim_data        = td.SimulationData.from_file(f'{save_dir}/{name}.hdf5')

# sim_data.simulation.plot_3d()
# print(sim_data.simulation.mediums)
# for monitor in sim_data.simulation.monitors:
#     print(monitor.name, monitor.type)

In [12]:
EV_Hfield = [
    # 8.42, 16.38  # D100L100
    # 10.35, 15.28 # D100 L25
    10.19, 5.35, 20.57 # D200L100
    # 10.24,             # D200 L25
]

for EV in EV_Hfield:
    process_single_monitor(
            monitor_name='DFT_out_plane_XZ',
            peak=ev_to_hz(EV),
            plane='Hxz',
            save_path=os.path.join('field_plots_' + name, f'{EV:.2f}eV_Hfield_XZ.png')
        )

(Hx.shape=(403, 203), expected=(203, 403))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(203, 403), vertical_axis shape=(203, 403)
  H-field array shape=(203, 403)
  Saved: 10.19eV_Hfield_XZ.png
✓ Completed DFT_out_plane_XZ

(Hx.shape=(403, 203), expected=(203, 403))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(203, 403), vertical_axis shape=(203, 403)
  H-field array shape=(203, 403)
  Saved: 5.35eV_Hfield_XZ.png
✓ Completed DFT_out_plane_XZ

(Hx.shape=(403, 203), expected=(203, 403))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(203, 403), vertical_axis shape=(203, 403)
  H-field array shape=(203, 403)
  Saved: 20.57eV_Hfield_XZ.png
✓ Completed DFT_out_plane_XZ



In [ ]:
EV_Hfield_offset = [
    # 8.37, 15.99        # D100L100
    # 10.17, 15.45       # D100 L25
    10.13, 5.398,           # D200L100
    # 10.23, 3.23, 18.33            # D200 L25

]

for EV in EV_Hfield_offset:
    process_single_monitor(
            monitor_name='DFT_out_plane_XZoffset',
            peak=ev_to_hz(EV),
            plane='Exz',
            save_path=os.path.join('field_plots_' + name, f'{EV:.2f}eV_Hfield_XZoffset.png')
        )

In [17]:
# Task
# EV_Hfield = 17.56
# peak_freq = ev_to_hz(EV_Hfield)

# # Define all plotting tasks
# tasks = [
#         # ('DFT_in_plane_slice0',       'Hxy', f'{EV_Hfield:.2f}eV_Hfield_IN_0.png'),
#         # ('DFT_in_plane_slice4.5',     'Hxy', f'{EV_Hfield:.2f}eV_Hfield_IN_4.5offset.png'),
#         # ('DFT_bottom_plane_slice0.5', 'Hxy', f'{EV_Hfield:.2f}eV_Hfield_BOT_0.5.png'),
#         # ('DFT_bottom_plane_slice14',  'Hxy', f'{EV_Hfield:.2f}eV_Hfield_BOT_14offset.png'),
#         ('DFT_out_plane_XZ',          'Hxz', f'{EV_Hfield:.2f}eV_Hfield_OUT_XZ.png'),
#         ('DFT_out_plane_XZoffset',    'Hxz', f'{EV_Hfield:.2f}eV_Hfield_OUT_XZoffset.png'),
#     ]

# # Process each monitor separately
# for i, (monitor, plane, filename) in enumerate(tasks, 1):
#     print(f"[{i}/{len(tasks)}]", end=" ")
#     process_single_monitor(
#         monitor_name=monitor,
#         peak=peak_freq,
#         plane=plane,
#         save_path=os.path.join('field_plots_' + name, filename)
#     )

[1/2] (Hx.shape=(253, 203), expected=(203, 253))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(203, 253), vertical_axis shape=(203, 253)
  H-field array shape=(203, 253)
  Saved: 17.56eV_Hfield_OUT_XZ.png
✓ Completed DFT_out_plane_XZ

[2/2] (Hx.shape=(253, 203), expected=(203, 253))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(203, 253), vertical_axis shape=(203, 253)
  H-field array shape=(203, 253)
  Saved: 17.56eV_Hfield_OUT_XZoffset.png
✓ Completed DFT_out_plane_XZoffset

